## Ensemble de estimadores que parte de subconjuntos del dataset balanceados y toma el resultado por votación (HARD y SOFT voting).

In [2]:
# Importaciones necesarias y semilla 
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, precision_score, recall_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif
import pandas as pd
import numpy as np

SEED = 777

# TODO Probar feature selection, 113 variables para 300 instancias es muy poco

In [3]:
# TODO Hacer para que ya saque las proporciones y lo haga todo
def create_balanced_datasets(negative_class: pd.DataFrame, positive_class: pd.DataFrame, 
                             n: int, chunk_size: int, SEED: int) -> list: 

    """
    Function that return n balanced datasets made of x negative class instances and
    y positive class instances. * For binary-classification only.

    negative_class = pandas dataframe with only negative instances 
    positive_class = pandas dataframe with only positive instances
    n = number of subsets to return
    chunk_size = number of elements of each subset
    """
    majority = negative_class if len(negative_class) > len(positive_class) else positive_class
    minority = negative_class if len(negative_class) < len(positive_class) else positive_class
    
    # Barajar clase mayoritaria
    majority = majority.sample(frac=1, random_state=SEED).reset_index(drop=True)
    
    balanced_datasets = []

    for i in range(n):
        majority_subset = majority[i*chunk_size:(i+1)*chunk_size]

        dataset = pd.concat([majority_subset, minority], axis=0)

         # Volvemos a barajar
        dataset = dataset.sample(frac=1, random_state=SEED).reset_index(drop=True)
    
        balanced_datasets.append(dataset)
    
    return balanced_datasets

In [4]:
# Cargamos los datos 
standard = pd.read_csv('../data/processed/data_trainstandard.csv')
standard_2 = pd.read_csv('../data/processed/data_teststandard.csv')
robust = pd.read_csv('../data/processed/data_trainrobust.csv')
robust_2 = pd.read_csv('../data/processed/data_testrobust.csv')
minmax = pd.read_csv('../data/processed/data_trainmin_max.csv')
minmax_2 = pd.read_csv('../data/processed/data_testmin_max.csv')
max_abs = pd.read_csv('../data/processed/data_trainmax_abs.csv')
max_abs2 = pd.read_csv('../data/processed/data_testmax_abs.csv')

data_standard = pd.concat([standard, standard_2], axis=0) # Normal 
data_robust = pd.concat([robust, robust_2], axis=0) # Robust
data_minmax = pd.concat([minmax, minmax_2], axis=0) # Min-Max
data_maxabs = pd.concat([max_abs, max_abs2], axis=0) # Max-abs

# Unimos los datasets para probar diferentes particionados con o sin estratificación...

Se probarán diferentes ensembles compuestos por `4 estimadores`, donde 3 estimadores tendrán `204 instancias` que representan a las clase negativa y `el restante tendrá 203`. Por la clase positiva tendrán `214 instancias`. Hemos balanceado los conjuntos.

### Datos Normal

In [5]:
# Hacemos split del dataset general 
X = data_standard.drop(columns=['Variable de Salida'])
y = data_standard['Variable de Salida']
X_train_global, X_test_global, y_train_global, y_test_global = train_test_split(
    X, y, 
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

# Conjunto de datos de entrenamiento. Lo unimos. Sobre este conjunto se harán los 
# subconjuntos balanceados. Se hacen los splits antes de todo, ya que la evaluación 
# será sonre el test, que mantiene la distribución que se encontrará en producción.

train_full = pd.concat([X_train_global, y_train_global], axis=1)
train_full.shape

(823, 113)

##### Proporciones

In [6]:
# Tomando un conjunto de datos, vemos como de distribuyen los valores de la variable a clasificar
train_full['Variable de Salida'].value_counts()

Variable de Salida
0    652
1    171
Name: count, dtype: int64

In [7]:
652 / 171 # 4 estimadores

3.8128654970760234

In [8]:
652 / 4 # De 163 instancias de la clase negativa

163.0

##### Creación de subconjuntos balanceados 

In [9]:
# Datos positivos y negativos
positive = train_full[train_full['Variable de Salida'] == 1]
negative = train_full[train_full['Variable de Salida'] == 0]

# Barajar negativos
negative = negative.sample(frac=1, random_state=SEED).reset_index(drop=True)

balanced_datasets = []
chunk_size = 163

for i in range(4):
    neg_subset = negative[i*chunk_size:(i+1)*chunk_size]
    
    dataset = pd.concat([neg_subset, positive], axis=0)

    # Volvemos a barajar
    dataset = dataset.sample(frac=1, random_state=SEED).reset_index(drop=True)
    
    balanced_datasets.append(dataset)


for i in balanced_datasets: print(i.shape)

(334, 113)
(334, 113)
(334, 113)
(334, 113)


In [10]:
# Con la función creada 
balanced_datasets = create_balanced_datasets(negative, positive, 4, 163, SEED)
for i in balanced_datasets: print(i.shape)

(334, 113)
(334, 113)
(334, 113)
(334, 113)


#### 1º Ensayo
Estimadores a probar: `Random Forest, Ada Boost, GradientBoostingClassfier`

In [11]:
estimators = [RandomForestClassifier(random_state=SEED, n_estimators=500, criterion='entropy', n_jobs=5), 
              AdaBoostClassifier(random_state=SEED, n_estimators=500),
               GradientBoostingClassifier(random_state=SEED, n_estimators=500), 
               SVC(random_state=SEED )]

trained_models = []

m = 0
for dataset in balanced_datasets:
    model = estimators[m]

    X = dataset.drop(columns=['Variable de Salida'])
    y = dataset['Variable de Salida']

    model.fit(X,y)
    trained_models.append(model)
    m += 1

In [12]:
for i in trained_models: print(i)

RandomForestClassifier(criterion='entropy', n_estimators=500, n_jobs=5,
                       random_state=777)
AdaBoostClassifier(n_estimators=500, random_state=777)
GradientBoostingClassifier(n_estimators=500, random_state=777)
SVC(random_state=777)


In [25]:
# Calculamos el f1-score de cada modelo HARD VOTING (predict)
metrics = []

for model in trained_models:
    y_pred = model.predict(X_test_global)
    #print(classification_report(y_test_global, y_pred))
    metrics.append(f1_score(y_test_global, y_pred, average='weighted'))

# Métricas individuales 
print(metrics)

# Creamos un y_pred que sea el resultado del hard voting de los demas modelos
y_preds = [model.predict(X_test_global) for model in trained_models]
y_pred_ensemble = np.array(y_preds).T
y_pred_ensemble = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=y_pred_ensemble)
print(f1_score(y_test_global, y_pred_ensemble, average='weighted'))

[0.581909664285116, 0.6079084861124521, 0.5467797231163801, 0.6284427546563468]
0.6596807968411291


In [ ]:
estimators = [RandomForestClassifier(random_state=SEED, n_estimators=1, criterion='entropy', 
                                     n_jobs=5, class_weight='balanced'), 
              AdaBoostClassifier(random_state=SEED, n_estimators=1),
               GradientBoostingClassifier(random_state=SEED, n_estimators=1), 
               SVC(random_state=SEED, probability=True, class_weight='balanced')]

trained_models = []

m = 0
for dataset in balanced_datasets:
    model = estimators[m]

    X = dataset.drop(columns=['Variable de Salida'])
    y = dataset['Variable de Salida']

    model.fit(X,y)
    trained_models.append(model)
    m += 1

probs = np.mean([m.predict_proba(X_test_global)[:,1] for m in trained_models], axis=0)

for t in np.arange(0.2, 0.6, 0.05):
    y_pred_t = (probs >= t).astype(int)
    r = recall_score(y_test_global, y_pred_t, pos_label=1)
    p = precision_score(y_test_global, y_pred_t, pos_label=1)
    f = f1_score(y_test_global, y_pred_t, pos_label=1)
    print(f"t={t:.2f} → Recall: {r:.2f} | Precision: {p:.2f} | F1: {f:.2f}")

t=0.20 → Recall: 1.00 | Precision: 0.21 | F1: 0.35
t=0.25 → Recall: 1.00 | Precision: 0.21 | F1: 0.35
t=0.30 → Recall: 0.88 | Precision: 0.23 | F1: 0.37
t=0.35 → Recall: 0.88 | Precision: 0.24 | F1: 0.37
t=0.40 → Recall: 0.88 | Precision: 0.24 | F1: 0.37
t=0.45 → Recall: 0.84 | Precision: 0.24 | F1: 0.38
t=0.50 → Recall: 0.65 | Precision: 0.29 | F1: 0.40
t=0.55 → Recall: 0.40 | Precision: 0.28 | F1: 0.33


In [ ]:
# Ver qué tan bien funciona cada modelo en sus propios datos de entrenamiento
for i, (model, dataset) in enumerate(zip(trained_models, balanced_datasets)):
    X = dataset.drop(columns=['Variable de Salida'])
    y = dataset['Variable de Salida']
    y_pred_train = model.predict(X)
    print(f"Modelo {i} - {type(model).__name__}")
    print(f"  F1 train clase 1: {f1_score(y, y_pred_train, pos_label=1):.3f}")
    print(f"  F1 test  clase 1: {f1_score(y_test_global, model.predict(X_test_global), pos_label=1):.3f}")

Modelo 0 - RandomForestClassifier
  F1 train clase 1: 0.827
  F1 test  clase 1: 0.364
Modelo 1 - AdaBoostClassifier
  F1 train clase 1: 0.688
  F1 test  clase 1: 0.354
Modelo 2 - GradientBoostingClassifier
  F1 train clase 1: 0.734
  F1 test  clase 1: 0.300
Modelo 3 - SVC
  F1 train clase 1: 0.739
  F1 test  clase 1: 0.305


In [ ]:
# Usar todos los datos de train balanceados juntos para seleccionar features
X_all_train = pd.concat([d.drop(columns=['Variable de Salida']) for d in balanced_datasets])
y_all_train = pd.concat([d['Variable de Salida'] for d in balanced_datasets])

# Probar con distintos k
for k in [20, 30, 40, 50]:
    selector = SelectKBest(mutual_info_classif, k=k)
    selector.fit(X_all_train, y_all_train)
    selected = X_all_train.columns[selector.get_support()]
    
    # Reentrenar y evaluar ensemble con esas features
    trained = []
    for dataset in balanced_datasets:
        X = dataset[selected]
        y = dataset['Variable de Salida']
        rf = RandomForestClassifier(n_estimators=200, random_state=SEED)
        rf.fit(X, y)
        trained.append(rf)
    
    probas = np.mean([m.predict_proba(X_test_global[selected])[:,1] for m in trained], axis=0)
    y_pred = (probas >= 0.35).astype(int)
    f = f1_score(y_test_global, y_pred, pos_label=1)
    r = recall_score(y_test_global, y_pred, pos_label=1)
    print(f"k={k} → F1: {f:.3f} | Recall: {r:.3f}")

k=20 → F1: 0.346 | Recall: 0.860
k=30 → F1: 0.373 | Recall: 0.953
k=40 → F1: 0.360 | Recall: 0.930
k=50 → F1: 0.366 | Recall: 0.953


In [ ]:
from sklearn.model_selection import ParameterGrid
from sklearn.feature_selection import SelectKBest, mutual_info_classif


param_grid = {
    'k': [25, 30, 35, 40],
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 5, 10, 15],
    'threshold': [0.30, 0.35, 0.40]
}

X_all_train = pd.concat([d.drop(columns=['Variable de Salida']) for d in balanced_datasets])
y_all_train = pd.concat([d['Variable de Salida'] for d in balanced_datasets])

results = []

for params in ParameterGrid(param_grid):
    # Selección de features
    selector = SelectKBest(mutual_info_classif, k=params['k'])
    selector.fit(X_all_train, y_all_train)
    selected = X_all_train.columns[selector.get_support()]
    
    # Entrenar ensemble
    trained = []
    for dataset in balanced_datasets:
        X = dataset[selected]
        y = dataset['Variable de Salida']
        rf = RandomForestClassifier(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            random_state=SEED, n_jobs=-1
        )
        rf.fit(X, y)
        trained.append(rf)
    
    # Evaluar
    probas = np.mean([m.predict_proba(X_test_global[selected])[:,1] for m in trained], axis=0)
    y_pred = (probas >= params['threshold']).astype(int)
    
    results.append({
        **params,
        'f1': f1_score(y_test_global, y_pred, pos_label=1),
        'recall': recall_score(y_test_global, y_pred, pos_label=1),
        'precision': precision_score(y_test_global, y_pred, pos_label=1)
    })

# Ver mejores resultados ordenados por recall (tu prioridad) con F1 aceptable
df_results = pd.DataFrame(results)
df_results[df_results['recall'] >= 0.85].sort_values('f1', ascending=False).head(15)

,k,max_depth,n_estimators,threshold,f1,recall,precision
37,30,NaN,100,0.35,0.374429,0.953488,0.232955
136,40,15.0,100,0.35,0.371041,0.953488,0.230337
133,40,10.0,500,0.35,0.371041,0.953488,0.230337
122,40,5.0,300,0.40,0.370732,0.883721,0.234568
64,30,15.0,100,0.35,0.369369,0.953488,0.229050
127,40,10.0,100,0.35,0.369369,0.953488,0.229050
61,30,10.0,500,0.35,0.369369,0.953488,0.229050
142,40,15.0,500,0.35,0.366972,0.930233,0.228571
55,30,10.0,100,0.35,0.366071,0.953488,0.226519
103,35,15.0,300,0.35,0.364444,0.953488,0.225275
